# Enhancing RoomFormer with DinoV3

## 1. Overall

RoomFormer: from the paper [Connecting the Dots: Floorplan Reconstruction Using Two-Level Queries](https://github.com/ywyue/RoomFormer/tree/main#preparation)  
DINOv3: from [DINOv3](https://github.com/facebookresearch/dinov3/tree/main)  
- We want to utilize the overall architecture of RoomFormer and further improving the results on floorplan reconstruction
- In RoomFormer, 3D point cloud are projected to a density map before going to the rest of the model
- Our approach:
    - Remove the projection phase, directly feeding the point cloud to the model by leveraging DINOv3 as an intermediate layer
    - Replace current encoder with [LitePT](https://github.com/prs-eth/LitePT), keeping decoder and two-level queries intact. 

## 2. Current progress

Note: it is hard to take a large step (i.e. directly adding a 3D point cloud preprocessing layer to RoomFormer), so the work is divided to smaller steps.

### 2.1 Retraining RoomFormer

We want to reproduce the result reported in the paper of RoomFormer. The actual retrained results are shown below:

|                              | room_prec ↑ | room_rec ↑ | corner_prec ↑ | corner_rec ↑ | angles_prec ↑ | angles_rec ↑ | room_f1 ↑ | corner_f1 ↑ | angles_f1 ↑ |
|------------------------------|-------------|------------|---------------|--------------|---------------|--------------|-----------|-------------|-------------|
Results in paper               |        97.9 |       96.7 |          89.1 |         85.3 |          83.0 |         79.5 |      97.3 |        87.2 |        81.2 |
Provided model                 |        97.9 |       96.8 |          89.2 |         85.3 |          83.0 |         79.4 |      97.4 |        87.3 |        81.2 |
Retrained model                |        89.2 |       87.0 |         77.08 |         72.0 |          68.0 |         63.5 |      88.0 |        74.5 |        65.7 |
Retrained model (~1000 epochs) |        91.5 |       89.9 |          79.9 |         76.7 |          72.2 |         69.3 |      90.7 |        78.2 |        70.7 |

As can be seen, the retrained results are nowhere near the reported ones. Currently waiting for response from the author on checking if there is anything settings/hyperparameter missing.

### 2.2 Adding DINOv3 feaures on top of density map

While waiting for the reply from the author(s), it's best we move on to the next step.  

DINOv3 is involved here as an enrichment layer for the density map input of RoomFormer. Specifically, we passed the density map through RoomFormer to produce a patch token of size $(batch\_size \times embed\_dim \times 16 \times 16)$, further pass it through a linear head (consists of a convolution layer and a batch norm layer) and interpolate to get the final feature map of size $(batch\_size \times 1 \times img\_size \times img\_size)$.

![Linear Head overall architecture](./imgs/modified_rf_v1_linear_head.png "Linear Head overall architecture")

The results are reported in the table below:

|                              | room_prec ↑ | room_rec ↑ | corner_prec ↑ | corner_rec ↑ | angles_prec ↑ | angles_rec ↑ | room_f1 ↑ | corner_f1 ↑ | angles_f1 ↑ |
|------------------------------|-------------|------------|---------------|--------------|---------------|--------------|-----------|-------------|-------------|
Results in paper               |        97.9 |       96.7 |          89.1 |         85.3 |          83.0 |         79.5 |      97.3 |        87.2 |        81.2 |
Provided model                 |        97.9 |       96.8 |          89.2 |         85.3 |          83.0 |         79.4 |      97.4 |        87.3 |        81.2 |
Retrained model                |        96.4 |       95.4 |          87.2 |         83.9 |          81.2 |         78.3 |      95.9 |        85.5 |        79.7 |
DINOv3-enhanced model (v1)     |        88.2 |       87.2 |          77.1 |         70.5 |          68.2 |         62.5 |      87.7 |        73.7 |        65.2 |
DINOv3-enhanced model (v2)     |        93.0 |       91.8 |          84.1 |         78.6 |          76.2 |         71.3 |      92.4 |        81.2 |        73.7 |


Overall, the results are discouraging. While both the original (retrained) and the modified models perform subpar to the provided model, DINOv3-enhanced model shown little to no improvement over the retrained model.

### 2.3 Adding DINOv3 features to ResNet output

The model is trained for ~1000 epochs, and the results are shown above. Overall, there are improvements compare to the retrained vanilla RoomFormer and DINOv3-enhanced model v1; however, we haven't reach the reported results of RoomFormer.

Running cross evaluation on scenecad for original paper and v2:
|                              | IoU ↑ | corner_prec ↑ | corner_rec ↑ | angles_prec ↑ | angles_rec ↑ | corner_f1 ↑ | angles_f1 ↑ |
|------------------------------|-------|---------------|--------------|---------------|--------------|-------------|-------------|
Results in paper               |  74.0 |          56.2 |         65.0 |          44.2 |         48.4 |        60.3 |        46.2 |
Provided model                 |  77.2 |          49.6 |         72.5 |          37.3 |         51.9 |        58.9 |        43.4 |
Retrained model (500 epochs)   |  75.5 |          46.8 |         71.9 |          34.9 |         51.1 |        56.7 |        41.5 |
DINOv3-enhanced model (v2)     |  72.4 |          66.1 |         65.4 |          53.1 |         52.9 |        65.7 |        53.0 |


As can be seen, for cross-data generalization evaluation (evaluate on different task category), DINO-v3-RF-v2 shown slightly better performance than the provided model. 

We will now try to adapt the model to only taking input from DINOv3, and adding a Linear mapping layer for DINOv3 to process point cloud data.

From the result above, it is safe to assume the benefits of adding enriched features to the output of ResNet (or skipping RoomFormer's ResNet feature extractor entirely in the feature). Following this approach, Esa suggested to use DINO on multiview images, that is, for each view (with some N visible corresponding 3D points), extract the view's feature patches with DINO and assigning each 3D point with the aggregated features (from multiple views). 

In [ ]:
import numpy as np


class ArgsTmp:
    def __init__(self):
        pass


args = ArgsTmp

args.lr = 2e-4
args.lr_backbone_names = ["backbone.0"]
args.lr_backbone = 2e-5
args.lr_linear_proj_names = ["sampling_offsets"]
args.lr_linear_proj_mult = 0.1
args.batch_size = 10
args.weight_decay = 1e-4
args.epochs = 500
args.lr_drop = [400]
args.clip_max_norm = 0.1

args.sgd = False

# backbone
args.backbone = "resnet50"
args.dilation = False
args.position_embedding = "sine"
args.position_embedding_scale = 2 * np.pi
args.num_feature_levels = 4

# Transformer
args.enc_layers = 6
args.dec_layers = 6
args.dim_feedforward = 1024
args.hidden_dim = 256
args.dropout = 0.1
args.nheads = 8  # , type=int,
# help="Number of attention heads inside the transformer's attentions")
args.num_queries = 800  # , type=int,
# help="Number of query slots (num_polys * max. number of corner per poly)")
args.num_polys = 20  # , type=int,
# help="Number of maximum number of room polygons")
args.dec_n_points = 4  # , type=int)
args.enc_n_points = 4  # , type=int)
args.query_pos_type = "sine"  # , type=str, choices=('static', 'sine', 'none'),
# help="Type of query pos in decoder - \
# 1. static: same setting with DETR and Deformable-DETR, the query_pos is the same for all layers \
# 2. sine: since embedding from reference points (so if references points update, query_pos also \
# 3. none: remove query_pos")
args.with_poly_refine = True  # , action='store_true',
# help="iteratively refine reference points (i.e. positional part of polygon queries)")
args.masked_attn = False  # , action='store_true',
# help="if true, the query in one room will not be allowed to attend other room")
args.semantic_classes = -1  # , type=int,
# help="Number of classes for semantically-rich floorplan:  \
# 1. default -1 means non-semantic floorplan \
# 2. 19 for Structured3D: 16 room types + 1 door + 1 window + 1 empty")

# loss
args.aux_loss = False  #', dest='aux_loss', action='store_true',
# help="Disables auxiliary decoding losses (loss at each layer)")

# matcher
args.set_cost_class = 2  # , type=float,
# help="Class coefficient in the matching cost")
args.set_cost_coords = 5  # , type=float,
# help="L1 coords coefficient in the matching cost")

# loss coefficients
args.cls_loss_coef = 2  # , type=float)
args.room_cls_loss_coef = 0.2  # , type=float)
args.coords_loss_coef = 5  # , type=float)
args.raster_loss_coef = 1  # , type=float)

# dataset parameters
args.dataset_name = "stru3d"
args.dataset_root = "data/stru3d_processed"  # , type=str)

args.output_dir = ("output",)
# help='path where to save, empty for no saving')
args.device = ("cuda",)
# help='device to use for training / testing')
args.seed = 42  # , type=int)
args.resume = ("",)  # help='resume from checkpoint')
args.start_epoch = 0  # , type=int, metavar='N',
# help='start epoch')
args.num_workers = 2  # , type=int)
args.job_name = "train_stru3d"  # , type=str)

args.wandb = False  # , action='store_true',# help='if added, initiate remote logging')

args.dinov3_repo = "dinov3"
args.dinov3_checkpoint = "checkpoints/dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
args.dinov3_n_last_layers = 4
args.lr_dinov3_head = 1e-3

args.device = "cuda"

args.num_points = 256


In [ ]:
from datasets import build_mixed_dataset as build_dataset

dataset_train = build_dataset(image_set="train", args=args)

In [ ]:
sample = dataset_train[0]

In [ ]:
sample["point_cloud"]["xyz"].shape

In [ ]:
sample["file_name"]

In [ ]:
from plyfile import PlyData

ply_path = "/home/hai/master-thesis/RoomFormer/data/stru3d_processed/train/scene_00400/point_cloud.ply"
plydata = PlyData.read(ply_path)
vertex = plydata["vertex"]

xyz = np.stack([vertex["x"], vertex["y"], vertex["z"]], axis=-1)

xyz.shape

In [ ]:
coords = xyz
coords[:, :2] = np.round(coords[:, :2] / 10) * 10.0
coords[:, 2] = np.round(coords[:, 2] / 100) * 100.0

unique_coords, unique_ind = np.unique(coords, return_index=True, axis=0)

unique_coords.shape

In [ ]:
import torch

# from torch import

DEVICE = "cuda"

dinov3 = torch.hub.load(
    args.dinov3_repo,
    "dinov3_vits16",
    source="local",
    weights=args.dinov3_checkpoint,
).to(DEVICE)

dinov3

In [ ]:
from datasets import build_poly_dataset as build_dataset

dataset_train = build_dataset(image_set="train", args=args)

In [ ]:
from torch import nn


class PositionalEncoding3D(nn.Module):
    def __init__(self, embed_dim, max_temp=10000):
        super().__init__()
        self.embed_dim = embed_dim
        self.max_temp = max_temp
        # We split the embedding dimension into 3 for X, Y, and Z
        self.channels = embed_dim // 3

    def forward(self, xyz):
        # xyz shape: [B, N, 3], assumed to be in [0, 1]
        device = xyz.device

        # Create the frequency scale
        dim_t = torch.arange(self.channels, dtype=torch.float32, device=device)
        dim_t = self.max_temp ** (2 * (dim_t // 2) / self.channels)

        # Reshape for broadcasting: [B, N, 3, 1] / [channels]
        pos = xyz.unsqueeze(-1) * 2 * torch.pi
        pos_scaled = pos / dim_t

        # Apply sine and cosine
        pos_sin = pos_scaled[:, :, :, 0::2].sin()
        pos_cos = pos_scaled[:, :, :, 1::2].cos()

        # Concatenate and flatten to [B, N, embed_dim]
        # Note: If embed_dim is not divisible by 6, you'll need to pad
        pos_feat = torch.cat([pos_sin, pos_cos], dim=-1).flatten(2)
        return pos_feat

In [ ]:
import torch

pc = sample["point_cloud"]["xyz"]

pe3d = PositionalEncoding3D(256)

pe3d(pc.unsqueeze(0)).shape

What to do next:
- roomformer_v3.py
  - Change positional encoding to PE3D above
  - (If possible) change ResNet to DINOv3, keep it taking images for now

- train_one_epoch and evaluate (engine_v3.py)
  - adapt to new dataset (point cloud & density map)

Esa's suggestion:
2. Compute DINOv3 features from the original RGB images, not from the density map.
Start with a frozen DINOv3 ViT-S/16 or ViT-B/16. Use the normalized spatial patch tokens from the final transformer layer. Do not use the CLS token or register tokens, because those are not spatially aligned with individual image regions. Please also use the official DINOv3 image normalization.

The DINO input dimensions should be multiples of 16. Preserve the image aspect ratio and keep an exact mapping between original RGB pixel coordinates and the resized/padded DINO image coordinates.
If you are using the Structured3D panoramas, please do not feed a complete equirectangular panorama directly into a standard ViT. Convert the panorama into calibrated perspective or cubemap views, or use another projection that retains an exact pixel-to-ray mapping. For ScanNet/SceneCAD perspective frames, the existing camera intrinsics and poses can be used directly.

3. Map the image patch features to 3D points.
For every 3D point and every candidate camera, project the point into the image using the camera intrinsics and world-to-camera pose. Accept the association only when:

the point is in front of the camera;
the projected pixel is inside the image;
its projected depth agrees with the RGB-D depth image;
it is not occluded by a closer surface.
After accounting for the DINO resize/padding transform, map the pixel to its corresponding 16 x 16 DINO patch and attach that patch token to the point.

If the point cloud preprocessing already stores the source frame and source pixel for every point, use those correspondences instead of projecting the fused point cloud back into all images.



TODO: Change RoomFormer input to include the required inputs
for DINO_BEV module. 

What to do (briefly, but also in a more detailed way)
Currently we are using a wrapper model that contains both 
RoomFormer (RF) and DINO_BEV (DBV), which passes the data 
through DBV, stack the output with the original data, and
pass the whole thing to RF. However, as is proven in v1 
and experiment on overfitting the model to a small subset,
it can be seen that this approach simply did not work. 
Reasons could be ResNets are trained on images, so passing
a 65-dim image like Tensor would have caused the problem.

So, I will now try to add DBV output to the output of RN, 
which has been shown to be affective in tag v1.9. Basically,
that means I will have to modify the forward function of RF
to also passing data to DBV, which means x has to be one 
batch and not just the image array like now. I will also 
have to change DBV to output multi-layered pyramid-like
patches that matches the size of multi-layered output of
ResNet, which basically means a ton of new parameters to be
added, new method of passing data around, some new Conv2d
layers (zero-initiated) to map the 
(batch_size, pca_outdim, 256, 256) to 
(batch_size, transformer_d_model, 16, 16), which is then 
added (TODO: ablation) to ResNet outputs before passing
to transformer.

In [ ]:
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

sample_path = os.path.join(
    "data", "Structured3D", "scene_00000", "2D_rendering", "485142", "panorama", "full"
)
sample_panorama = os.path.join(sample_path, "rgb_coldlight.png")
sample_depth = os.path.join(sample_path, "depth.png")

depth = cv2.imread(sample_depth, cv2.IMREAD_UNCHANGED)
# plt.imshow(sample_panorama)

In [ ]:
import numpy as np

data_dir = os.path.join("data", "Structured3D")
scenes = os.listdir(data_dir)
sample_scenes = np.random.choice(scenes, 10).tolist() + ["scene_00000"]

sample_scenes

In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import py360convert
from tqdm import tqdm

import matplotlib.pyplot as plt

In [ ]:
layout = "data/Structured3D/scene_00000/2D_rendering/485142/panorama/layout.txt"
layout_pts = []
with open(layout, "r") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 2:
            layout_pts.append([float(parts[0]), float(parts[1])])

layout_pts = np.array(layout_pts, dtype=np.float32)

img = cv2.imread(
    "data/Structured3D/scene_00000/2D_rendering/485142/panorama/full/rgb_coldlight.png",
    cv2.IMREAD_COLOR,
)
# for i, (u, v) in enumerate(layout_pts):
#     color = (0, 255, 0) # if (i % 2 == 0) else (0, 0, 255)  # Green=Ceiling, Red=Floor
#     cv2.circle(img, (int(u), int(v)), 4, color, -1)
plt.figure(figsize=(10, 20))
plt.imshow(img)
# layout_pts

In [ ]:
depth = cv2.imread(
    "data/Structured3D/scene_00000/2D_rendering/485142/panorama/full/depth.png",
    cv2.IMREAD_UNCHANGED,
)

In [ ]:
mask = np.zeros_like(img)

for u, v in layout_pts:
    for x in range(int(v) - 5, int(v) + 5):
        for y in range(int(u) - 5, int(u) + 5):
            mask[int(x), int(y)] = 255

plt.imshow(mask)

In [ ]:
faces = py360convert.e2c(img, face_w=256, cube_format="dict")
masks = py360convert.e2c(mask, face_w=256, cube_format="dict")

In [ ]:
plt.figure(figsize=(10, 20))
plt.subplot(1, 2, 1)
plt.imshow(faces["R"])
plt.subplot(1, 2, 2)
plt.imshow(masks["R"])
plt.show()

pipeline:

py360convert -> cube map of panorama -> DINO 

point cloud -> which are visible in each face -> DINO features to each point

to know which point cloud is visible in each face -> 3D-to-2D map -> K [R | t] @ point 

In [ ]:
from plyfile import PlyData

print("--- STEP 1: Loading PLY Point Cloud ---")

ply_path = "data/Structured3D/scene_00000/point_cloud.ply"

plydata = PlyData.read(ply_path)
vertex = plydata["vertex"]

# Extract 3D coordinates (X, Y, Z)
pts_3d_world = np.stack([vertex["x"], vertex["y"], vertex["z"]], axis=1).astype(
    np.float32
)

# Extract RGB Colors (handles 0-255 uint8 or 0.0-1.0 float format)
if "red" in vertex.data.dtype.names:
    colors_r = vertex["red"]
    colors_g = vertex["green"]
    colors_b = vertex["blue"]

    # Normalize uint8 (0-255) to float (0-1) for matplotlib
    if colors_r.max() > 1.0:
        colors_world = (
            np.stack([colors_r, colors_g, colors_b], axis=1).astype(np.float32) / 255.0
        )
    else:
        colors_world = np.stack([colors_r, colors_g, colors_b], axis=1).astype(
            np.float32
        )
else:
    # Default fallback color (gray) if PLY lacks RGB channels
    print("no rgb")
    colors_world = np.ones_like(pts_3d_world) * 0.5

print(f"Loaded Point Cloud: {pts_3d_world.shape[0]} points")

img = cv2.imread(
    "data/Structured3D/scene_00000/2D_rendering/485145/panorama/full/rgb_coldlight.png",
    cv2.IMREAD_COLOR,
)
depth = cv2.imread(
    "data/Structured3D/scene_00000/2D_rendering/485145/panorama/full/depth.png",
    cv2.IMREAD_UNCHANGED,
)

plt.figure(figsize=(10, 20))
plt.imshow(img)
print(img.shape)

In [ ]:
# ==============================================================================
# STEP 2: Extract 6 Cubemap Faces using py360convert
# ==============================================================================
print("--- STEP 2: Extracting Cubemap Faces ---")

face_w = 512  # Width/Height of each square cubemap face

# py360convert e2c produces dict with keys: 'F', 'R', 'B', 'L', 'U', 'D'
cubemap_dict = py360convert.e2c(img, face_w=face_w, cube_format="dict")
depth_dict = py360convert.e2c(depth[:, :, None], face_w=face_w, cube_format="dict")

# Verify extraction visually
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
fig.suptitle("Extracted 6 Cubemap Faces", fontsize=14)
face_order = ["U", "F", "R", "L", "B", "D"]
for ax, key in zip(axes.flat, face_order):
    ax.imshow(cubemap_dict[key])  # cv2.cvtColor(cubemap_dict[key], cv2.COLOR_BGR2RGB))
    ax.set_title(f"Face: {key}")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
print("--- STEP 3: Computing Camera Parameters (K, R, T) ---")


def get_cubemap_intrinsics(width):
    """
    Computes Intrinsic Matrix K for a 90-degree FOV square cubemap face.
    Focal length f = W / (2 * tan(90/2)) = W / 2
    """
    f = width / 2.0
    cx, cy = width / 2.0, width / 2.0
    K = np.array([[f, 0, cx], [0, f, cy], [0, 0, 1]], dtype=np.float32)
    return K


# Intrinsic matrix for face_w x face_w face
K_face = get_cubemap_intrinsics(face_w)

# Define Rotation matrices for each face relative to standard camera coordinate frame
# OpenCV Frame: +X Right, +Y Down, +Z Forward
ROTATIONS = {
    "F": np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]], dtype=np.float32),  # Yaw   0°
    "R": np.array([[0, 0, -1], [0, 1, 0], [1, 0, 0]], dtype=np.float32),  # Yaw +90°
    "B": np.array([[-1, 0, 0], [0, 1, 0], [0, 0, -1]], dtype=np.float32),  # Yaw 180°
    "L": np.array([[0, 0, 1], [0, 1, 0], [-1, 0, 0]], dtype=np.float32),  # Yaw -90°
    "D": np.array([[1, 0, 0], [0, 0, -1], [0, 1, 0]], dtype=np.float32),  # Pitch +90°
    "U": np.array([[1, 0, 0], [0, 0, 1], [0, -1, 0]], dtype=np.float32),  # Pitch -90°
}


# Base transformation mapping World (+Z Up, +Y Look) -> Standard OpenCV Frame (+Y Down, +Z Look)
R_base = np.array([[1, 0, 0], [0, 0, -1], [0, 1, 0]], dtype=np.float32)

# Camera Translation
with open(
    "data/Structured3D/scene_00000/2D_rendering/485145/panorama/camera_xyz.txt", "rb"
) as f:
    T_world = [float(v) for v in f.readline().strip().split()]
T_world = np.array(T_world, dtype=np.float32)

print("Intrinsic Matrix K:")
print(K_face)

print("camera center T:")
print(T_world)

In [ ]:
def radial_to_planar_depth(depth_radial, K):
    """Converts 360 radial depth map into pinhole planar Z-depth map."""
    h, w = depth_radial.shape[:2]
    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]

    x, y = np.meshgrid(np.arange(w), np.arange(h))
    x_dir = (x - cx) / fx
    y_dir = (y - cy) / fy
    ray_length = np.sqrt(x_dir**2 + y_dir**2 + 1.0)

    return (depth_radial.squeeze() / ray_length).astype(np.float32)


# Precompute Planar Z-Depth for each cubemap face
K_face = np.array(
    [[face_w / 2.0, 0, face_w / 2.0], [0, face_w / 2.0, face_w / 2.0], [0, 0, 1]],
    dtype=np.float32,
)

depth_planar_dict = {
    key: radial_to_planar_depth(depth_dict[key], K_face) for key in cubemap_dict.keys()
}

In [ ]:
print("\n--- STEP 4: Filtering 3D Points for Each Face ---")


def get_points_in_face(
    pts_3d, colors, K, R_face, R_base, T_cam, depth, img_w, img_h, depth_tol=10.0
):
    """
    Transforms 3D points into camera space and determines which points project
    inside the image boundaries [0, img_w] x [0, img_h].
    """
    # 1. Combined World-to-Camera Rotation Matrix
    R_total = R_face @ R_base

    # 2. Transform 3D World Points -> 3D Camera Points
    pts_cam = (R_total @ (pts_3d - T_cam).T).T  # (N, 3)

    # 4. Project onto Image Plane: [u, v, 1] = K * [X/Z, Y/Z, 1]
    pts_proj = (K @ (pts_cam / pts_cam[:, 2:3]).T).T
    u = pts_proj[:, 0]
    v = pts_proj[:, 1]

    # 3. Filter points in front of the camera (Z > 0)
    valid_z = pts_cam[:, 2] > 0.1

    # 5. Filter points strictly inside pixel bounds
    in_bounds = (u >= 0) & (u < img_w) & (v >= 0) & (v < img_h) & valid_z

    u_valid = u[in_bounds].astype(int)
    v_valid = v[in_bounds].astype(int)
    z_point = pts_cam[in_bounds, 2]

    # 6. DEPTH VERIFICATION (Occlusion / Distance Agreement)
    # Query corresponding planar depth from face depth map
    z_map_expected = depth[v_valid, u_valid]

    # Keep points whose 3D depth matches the face depth map within tolerance
    depth_match = (np.abs(z_point - z_map_expected) <= depth_tol) & (z_map_expected > 0)

    # Combine masks
    final_mask = np.zeros_like(in_bounds, dtype=bool)
    final_indices = np.where(in_bounds)[0][depth_match]
    final_mask[final_indices] = True

    return final_mask, u[final_mask], v[final_mask]


# Plot filtered 3D points projected onto each cubemap face
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle(
    "3D Points Correctly Masked & Projected onto Each Cubemap Face", fontsize=14
)

for ax, face_key in zip(axes.flat, face_order):
    R_face = ROTATIONS[face_key]

    depth = depth_planar_dict[face_key]
    # Find mask of 3D points that lie inside this face
    mask, u_pts, v_pts = get_points_in_face(
        pts_3d_world,
        colors_world,
        K_face,
        R_face,
        R_base,
        T_world,
        depth,
        face_w,
        face_w,
    )

    # Render background image
    img_rgb = cubemap_dict[face_key]  # cv2.cvtColor(, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)

    # Overlay projected points (subsampled for clarity)
    ax.scatter(u_pts, v_pts, c=colors_world[mask], s=3, edgecolors="none")
    ax.set_title(f"Face '{face_key}': {np.sum(mask)} Points In View")
    ax.set_xlim([0, face_w])
    ax.set_ylim([face_w, 0])  # Invert y-axis to match pixel space
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Verification complete! Points match their underlying cubemap faces.")

In [ ]:
mask.shape

In [ ]:
sample_scene = os.path.join("data", "Structured3D", "scene_00000", "2D_rendering")
rooms = [
    os.path.join(sample_scene, r, "panorama", "full") for r in os.listdir(sample_scene)
]
panoramas = [os.path.join(r, "rgb_coldlight.png") for r in rooms]
depths = [os.path.join(r, "depth.png") for r in rooms]

for panorama, depth in zip(panoramas, depths):
    assert os.path.exists(panorama), f"{panorama} not found"
    assert os.path.exists(depth), f"{depth} not found"


In [ ]:
from torchvision.transforms import v2


def make_transform(resize_size: int = 256):
    # to_tensor = v2.ToImage()
    resize = v2.Resize((resize_size, resize_size), antialias=True)
    to_float = v2.ToDtype(torch.float32, scale=True)
    normalize = v2.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )
    return v2.Compose([resize, to_float, normalize])


transform = make_transform()

How is DINO features assigned to each visible point?

In DITR, projected points (u_pts, v_pts) are divided by patch_size (16, 16) and round down to obtain the point's corresponding feature block (Bx384x1x1)

Also need to implement generate_density to work with (Bx64x256x256) (RoomFormer's function only return image)

In [49]:
import numpy as np


class ArgsTmp:
    def __init__(self):
        pass


args = ArgsTmp

args.lr = 2e-4
args.lr_backbone_names = ["backbone.0"]
args.lr_backbone = 2e-5
args.lr_linear_proj_names = ["sampling_offsets"]
args.lr_linear_proj_mult = 0.1
args.batch_size = 10
args.weight_decay = 1e-4
args.epochs = 500
args.lr_drop = [400]
args.clip_max_norm = 0.1

args.sgd = False

# backbone
args.backbone = "resnet50"
args.dilation = False
args.position_embedding = "sine"
args.position_embedding_scale = 2 * np.pi
args.num_feature_levels = 4

# Transformer
args.enc_layers = 6
args.dec_layers = 6
args.dim_feedforward = 1024
args.hidden_dim = 256
args.dropout = 0.1
args.nheads = 8  # , type=int,
# help="Number of attention heads inside the transformer's attentions")
args.num_queries = 800  # , type=int,
# help="Number of query slots (num_polys * max. number of corner per poly)")
args.num_polys = 20  # , type=int,
# help="Number of maximum number of room polygons")
args.dec_n_points = 4  # , type=int)
args.enc_n_points = 4  # , type=int)
args.query_pos_type = "sine"  # , type=str, choices=('static', 'sine', 'none'),
# help="Type of query pos in decoder - \
# 1. static: same setting with DETR and Deformable-DETR, the query_pos is the same for all layers \
# 2. sine: since embedding from reference points (so if references points update, query_pos also \
# 3. none: remove query_pos")
args.with_poly_refine = True  # , action='store_true',
# help="iteratively refine reference points (i.e. positional part of polygon queries)")
args.masked_attn = False  # , action='store_true',
# help="if true, the query in one room will not be allowed to attend other room")
args.semantic_classes = -1  # , type=int,
# help="Number of classes for semantically-rich floorplan:  \
# 1. default -1 means non-semantic floorplan \
# 2. 19 for Structured3D: 16 room types + 1 door + 1 window + 1 empty")

# loss
args.aux_loss = False  #', dest='aux_loss', action='store_true',
# help="Disables auxiliary decoding losses (loss at each layer)")

# matcher
args.set_cost_class = 2  # , type=float,
# help="Class coefficient in the matching cost")
args.set_cost_coords = 5  # , type=float,
# help="L1 coords coefficient in the matching cost")

# loss coefficients
args.cls_loss_coef = 2  # , type=float)
args.room_cls_loss_coef = 0.2  # , type=float)
args.coords_loss_coef = 5  # , type=float)
args.raster_loss_coef = 1  # , type=float)

# dataset parameters
args.dataset_name = "stru3d"
args.dataset_root = "data/stru3d_processed"  # , type=str)

args.output_dir = ("output",)
# help='path where to save, empty for no saving')
args.device = ("cuda",)
# help='device to use for training / testing')
args.seed = 42  # , type=int)
args.resume = ("",)  # help='resume from checkpoint')
args.start_epoch = 0  # , type=int, metavar='N',
# help='start epoch')
args.num_workers = 2  # , type=int)
args.job_name = "train_stru3d"  # , type=str)

args.wandb = False  # , action='store_true',# help='if added, initiate remote logging')

args.dinov3_repo = "dinov3"
args.dinov3_checkpoint = "checkpoints/dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
args.dinov3_n_last_layers = 4
args.lr_dinov3_head = 1e-3

args.device = "cuda"

args.num_points = 256

args.dino_bev_aggregation = "random"
args.subset_length = 10

args.dinov3_model = "dinov3_vits16"
args.pca_outdim = 64
from datasets import build_cube_poly as build_dataset

dataset_train = build_dataset(mode="train", args=args)
from torch.utils.data import RandomSampler, BatchSampler, DataLoader

from util.poly_ops import pad_gt_polys

from models.roomformer_v3 import build

model, pca, criterion = build(args)
model.to(args.device)

sampler_train = RandomSampler(dataset_train)

batch_sampler_train = BatchSampler(sampler_train, args.batch_size, drop_last=True)
FACES = sorted(["U", "F", "R", "L", "B", "D"])


def batch_collator(batch):
    """
    A batch collator that does something.
    """
    scene_ids = [x["image_id"] for x in batch]
    samples = [x["image"] for x in batch]
    gt_instances = [x["instances"] for x in batch]
    room_targets = pad_gt_polys(
        gt_instances, model.num_queries_per_poly, device="cpu"
    )

    batched_room_faces = []  # [[x['cubes'][room] for room in x['cubes']] for x in batch]
    batched_room_depths = []
    batched_point_clouds = []
    batched_prj_points = []
    batched_masks = []

    for x in batch:
        rooms = sorted(x["cubes"].keys())
        tmp_cube = []
        tmp_depth = []
        tmp_prj_points = []
        tmp_masks = []
        for room in rooms:
            room_face = []
            room_depth = []
            room_prj_points = []
            room_mask = []
            for face in FACES:
                room_face.append(torch.moveaxis(x["cubes"][room][face], -1, 0))
                room_depth.append(torch.moveaxis(x["depths"][room][face], -1, 0))
                room_prj_points.append(x["project_points"][room][face])
                room_mask.append(x["masks"][room][face])
            tmp_cube.append(torch.stack(room_face))
            tmp_depth.append(torch.stack(room_depth))
            tmp_prj_points.append(room_prj_points)
            tmp_masks.append(room_mask)

        batched_room_faces.append(
            torch.flatten(torch.stack(tmp_cube), start_dim=0, end_dim=1)
        )
        batched_room_depths.append(
            torch.flatten(torch.stack(tmp_depth), start_dim=0, end_dim=1)
        )
        batched_prj_points.append(tmp_prj_points)
        batched_masks.append(tmp_masks)
        batched_point_clouds.append(x["point_cloud"])

    return (
        scene_ids,
        samples,
        gt_instances,
        room_targets,
        batched_room_faces,
        batched_room_depths,
        batched_point_clouds,
        batched_prj_points,
        batched_masks,
    )


train_loader = DataLoader(
    dataset_train, batch_sampler=batch_sampler_train, collate_fn=batch_collator
)
import torch

from torch.utils.data import RandomSampler, BatchSampler, DataLoader
from models.dino_bev import load_DINO, make_transform, extract_patch_grid
from datasets import build_cube_poly as build_dataset

dataset_train = build_dataset(mode="train", args=args)

FACES = sorted(["U", "F", "R", "L", "B", "D"])


dino = load_DINO(args.dinov3_repo, args.dinov3_checkpoint)
transform = make_transform()
# dino = load_DINO(repo=args.dinov3_repo, checkpoint=args.dinov3_checkpoint)


class DatasetWrapper(torch.utils.data.Dataset):
    def __init__(self, parent: torch.utils.data.Dataset):
        super().__init__()
        self.parent = parent

    def __len__(self):
        return self.parent.__len__()

    def __getitem__(self, idx):
        sample = self.parent[idx]
        rooms = sample["cubes"].keys()

        room_faces = []
        for room in rooms:
            for face in FACES:
                room_faces.append(
                    torch.moveaxis(sample["cubes"][room][face], -1, 0).to(args.device)
                )

        room_faces = torch.stack(room_faces)
        patches = extract_patch_grid(
            dino, transform(room_faces)
        )  # .repeat(1, 1, 16, 16)
        return patches


def tmp_collate(batch):
    return torch.cat(batch, dim=0)


dataset_train_mod = DatasetWrapper(dataset_train)
sampler_train = RandomSampler(dataset_train_mod)

batch_sampler_train_mod = BatchSampler(sampler_train, args.batch_size, drop_last=True)
train_loader_mod = DataLoader(
    dataset_train_mod, batch_sampler=batch_sampler_train_mod, collate_fn=tmp_collate
)
# pca = PCAWrapper()
pca.fit(train_loader_mod)
for batch in train_loader:
    (
        scene_ids,
        samples,
        gt_instances,
        room_targets,
        batched_room_faces,
        batched_room_depths,
        batched_point_clouds,
        batched_prj_points,
        batched_masks,
    ) = batch
    break


def _batch_to(batch, device="cpu"):
    (
        scene_ids,
        samples,
        gt_instances,
        room_targets,
        batched_room_faces,
        batched_room_depths,
        batched_point_clouds,
        batched_prj_points,
        batched_masks,
    ) = batch
    samples = [s.to(device, non_blocking=True) for s in samples]
    gt_instances = [s.to(device) for s in gt_instances]
    room_targets = [
        {
            "coords": t["coords"].to(device, non_blocking=True),
            "labels": t["labels"].to(device, non_blocking=True),
            "lengths": t["lengths"].to(device, non_blocking=True),
            "room_labels": t["room_labels"].to(device, non_blocking=True),
        }
        for t in room_targets
    ]
    batched_room_faces = [
        faces.to(device, non_blocking=True) for faces in batched_room_faces
    ]
    batched_room_depths = [
        faces.to(device, non_blocking=True) for faces in batched_room_depths
    ]
    batched_point_clouds = [
        faces.to(device, non_blocking=True) for faces in batched_point_clouds
    ]

    for scene_idx in range(len(batched_prj_points)):
        for room_idx in range(len(batched_prj_points[scene_idx])):
            for face_idx in range(len(FACES)):
                batched_prj_points[scene_idx][room_idx][face_idx] = batched_prj_points[
                    scene_idx
                ][room_idx][face_idx].to(device, non_blocking=True)

                batched_masks[scene_idx][room_idx][face_idx] = batched_masks[scene_idx][
                    room_idx
                ][face_idx].to(device, non_blocking=True)

    return (
        scene_ids,
        samples,
        gt_instances,
        room_targets,
        batched_room_faces,
        batched_room_depths,
        batched_point_clouds,
        batched_prj_points,
        batched_masks,
    )


(
    scene_ids,
    samples,
    gt_instances,
    room_targets,
    batched_room_faces,
    batched_room_depths,
    batched_point_clouds,
    batched_prj_points,
    batched_masks,
) = _batch_to(batch, args.device)
batch = (
    scene_ids,
    samples,
    gt_instances,
    room_targets,
    batched_room_faces,
    batched_room_depths,
    batched_point_clouds,
    batched_prj_points,
    batched_masks,
)

loading annotations into memory...
Done (t=0.22s)
creating index...
index created!


/home/hai/.miniconda3/envs/venv/lib/python3.11/site-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hai/.miniconda3/envs/venv/lib/python3.11/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


loading annotations into memory...
Done (t=0.22s)
creating index...
index created!


0it [00:28, ?it/s]


In [2]:
dbv = model.dino_multilayer.dino_bev

out, mask, cnt, agree = dbv(
    batched_room_faces, batched_masks, batched_point_clouds, batched_prj_points
)

print(out.shape)

torch.Size([10, 64, 256, 256])


In [3]:
model.num_feature_levels
len(model.backbone.strides)
model.transformer.d_model
model.backbone.num_channels

[512, 1024, 2048]

In [4]:
from torch import nn


class DINOMultiLayeredAdapter(nn.Module):
    def __init__(
        self,
        dino_bev: nn.Module,
        num_feature_levels: int,
        num_backbone_outs: int,
        in_channels: int,  # args.pca_outdim
        hidden_dim: int,
        out_width: list[int], # 
    ):
        super().__init__()
        self.dino_bev = dino_bev
        patch_size = [hidden_dim // x for x in out_width]  # should be 32, 16, 8, 4
        input_proj = []
        for i in range(num_backbone_outs):
            input_proj.append(
                nn.Sequential(
                    nn.Conv2d(
                        in_channels,
                        hidden_dim,
                        kernel_size=patch_size[i],
                        stride=patch_size[i],
                    ),
                    nn.GroupNorm(32, hidden_dim),
                )
            )
        for i in range(num_backbone_outs, num_feature_levels):
            input_proj.append(
                nn.Sequential(
                    nn.Conv2d(
                        in_channels,
                        hidden_dim,
                        kernel_size=patch_size[i],
                        stride=patch_size[i],
                    ),
                    nn.GroupNorm(32, hidden_dim),
                )
            )
            in_channels = hidden_dim
        self.input_proj = nn.ModuleList(input_proj)

    def forward(self, batch):
        (
            _,
            _,
            _,
            _,
            batched_room_faces,
            _,
            batched_point_clouds,
            batched_prj_points,
            batched_masks,
        ) = batch
        batch_scene_bev, batch_scene_mask, batch_scene_cnt, batch_scene_agree = (
            self.dino_bev(
                batched_room_faces,
                batched_masks,
                batched_point_clouds,
                batched_prj_points,
            )
        )
        out = [layer(batch_scene_bev) for layer in self.input_proj]
        return out

dmls = DINOMultiLayeredAdapter(
    dino_bev=dbv,
    num_feature_levels=model.num_feature_levels,
    num_backbone_outs=len(model.backbone.strides),
    in_channels=args.pca_outdim,
    hidden_dim=model.transformer.d_model,
    out_width=[32,16,8,4]
).to(args.device)

out = dmls(batch)


In [5]:
print(len(out))
for o in out:
    print(o.shape)

4
torch.Size([10, 256, 32, 32])
torch.Size([10, 256, 16, 16])
torch.Size([10, 256, 8, 8])
torch.Size([10, 256, 4, 4])


In [6]:
model(batch)

/home/hai/.miniconda3/envs/venv/lib/python3.11/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4215.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


{'pred_logits': tensor([[[-4.5277, -4.2345, -4.3128,  ..., -4.3843, -4.8202, -4.7790],
          [-5.0200, -5.2111, -5.0187,  ..., -4.7095, -4.7572, -5.0404],
          [-4.6705, -5.3750, -4.3125,  ..., -4.6421, -4.7908, -4.9196],
          ...,
          [-4.3556, -4.7400, -4.7672,  ..., -4.8138, -4.8908, -4.7310],
          [-4.3509, -4.1452, -4.3274,  ..., -4.9444, -5.0814, -4.8329],
          [-5.0861, -4.5793, -4.4651,  ..., -4.7543, -4.3674, -4.4274]],
 
         [[-4.8081, -3.9814, -4.5729,  ..., -4.7018, -4.8147, -5.0696],
          [-4.6242, -4.8116, -4.5476,  ..., -4.3299, -4.6045, -4.9299],
          [-5.0441, -3.9840, -4.5033,  ..., -4.4937, -4.1896, -4.8771],
          ...,
          [-4.5388, -4.9840, -4.5414,  ..., -4.7765, -5.1559, -5.3159],
          [-5.0260, -4.6584, -4.5726,  ..., -4.4140, -4.5259, -5.0180],
          [-4.3672, -4.6474, -4.9172,  ..., -4.6990, -5.0688, -4.4400]],
 
         [[-4.6433, -4.4189, -4.2478,  ..., -3.9274, -4.2077, -4.5287],
          [-4

In [51]:
import os
from pathlib import Path

from torch import nn
from models import build_model_v3 as build

def _load_pretrained_weights(
    model: nn.Module, old_weights_path: str | Path, freeze: bool = True
) -> nn.Module:
    assert os.path.exists(old_weights_path)
    model_dict = model.state_dict()

    old_weights = torch.load(old_weights_path, weights_only=False)
    filtered_weights = {}
    for k, v in old_weights["model"].items():
        if k in model_dict and v.shape == model_dict[k].shape:
            filtered_weights[k] = v

    model_dict.update(filtered_weights)
    model.load_state_dict(model_dict)

    if freeze:
        for name, param in new_model.named_parameters():
            if name in filtered_weights:
                print(name)
                param.requires_grad_(False)
            else:
                param.requires_grad_(True)
    return model


# old_model = torch.load('/home/hai/master-thesis/RoomFormer/checkpoints/roomformer_stru3d.pth', weights_only=False)
new_model, _, _ = build(args)
new_model = _load_pretrained_weights(
    new_model, "/home/hai/master-thesis/RoomFormer/checkpoints/roomformer_stru3d.pth"
)


/home/hai/.miniconda3/envs/venv/lib/python3.11/site-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hai/.miniconda3/envs/venv/lib/python3.11/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


transformer.level_embed
transformer.encoder.layers.0.self_attn.sampling_offsets.weight
transformer.encoder.layers.0.self_attn.sampling_offsets.bias
transformer.encoder.layers.0.self_attn.attention_weights.weight
transformer.encoder.layers.0.self_attn.attention_weights.bias
transformer.encoder.layers.0.self_attn.value_proj.weight
transformer.encoder.layers.0.self_attn.value_proj.bias
transformer.encoder.layers.0.self_attn.output_proj.weight
transformer.encoder.layers.0.self_attn.output_proj.bias
transformer.encoder.layers.0.norm1.weight
transformer.encoder.layers.0.norm1.bias
transformer.encoder.layers.0.linear1.weight
transformer.encoder.layers.0.linear1.bias
transformer.encoder.layers.0.linear2.weight
transformer.encoder.layers.0.linear2.bias
transformer.encoder.layers.0.norm2.weight
transformer.encoder.layers.0.norm2.bias
transformer.encoder.layers.1.self_attn.sampling_offsets.weight
transformer.encoder.layers.1.self_attn.sampling_offsets.bias
transformer.encoder.layers.1.self_attn.a

In [42]:
flag = torch.randn([572]) > 0.5
feat = torch.randn([572, 64])

out = feat * flag[:,None]

print(flag[:5])

tensor([False, False, False, False,  True])


In [43]:
print(out[:5])

tensor([[ 0.0000, -0.0000,  0.0000,  0.0000,  0.0000,  0.0000, -0.0000,  0.0000,
         -0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000, -0.0000,  0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000,
          0.0000, -0.0000,  0.0000, -0.0000,  0.0000,  0.0000,  0.0000, -0.0000,
          0.0000,  0.0000,  0.0000, -0.0000,  0.0000, -0.0000,  0.0000, -0.0000,
          0.0000, -0.0000,  0.0000,  0.0000,  0.0000,  0.0000, -0.0000, -0.0000,
          0.0000, -0.0000,  0.0000,  0.0000, -0.0000,  0.0000,  0.0000,  0.0000,
          0.0000, -0.0000,  0.0000, -0.0000, -0.0000, -0.0000,  0.0000, -0.0000],
        [ 0.0000,  0.0000,  0.0000, -0.0000,  0.0000, -0.0000, -0.0000,  0.0000,
          0.0000, -0.0000,  0.0000,  0.0000, -0.0000, -0.0000,  0.0000, -0.0000,
          0.0000, -0.0000, -0.0000, -0.0000, -0.0000,  0.0000,  0.0000, -0.0000,
         -0.0000,  0.0000, -0.0000, -0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         -0.0000,  0.0000, 